# AI-Powered Promotional Copy Generation - Case Study

Welcome to the case study! Your goal is to build an automated pipeline that takes promotion data and generates tailored marketing copy using Large Language Models (LLMs).

### Requirements:
1. **Input Data**: Read the `data/promotions.csv`.
2. **Style Guides**: Read the corresponding style guidelines from the `styles/` folder for each client.
3. **Generation**: For **each item** in the CSV, generate **3 examples**. Each example must include:
   - A `simplified` version of the copy.
   - A `complicated` version of the copy.
4. **LLM API**: Use the provided `llm.chat()` function from `src.llm` to simulate LLM calls.
5. **Verification (Optional/Bonus)**: Write a verification step to ensure the generated output follows the constraints in the `.md` files (e.g., price is mentioned, length limit is respected).

In [10]:
import pandas as pd
import json
import os
import sys

# Allow importing from src
sys.path.append('src')
from llm import llm

### 1. Load the Data & Style Guides

In [11]:
# Load promotion data
df = pd.read_csv('data/promotions.csv')
display(df.head())


,sku,title,promo_start,promo_end,orig_price,promo_price,category,client,description,size,max_copy_chars
0,SKU_TECH_001,TechNet Dual-Band Router,2026/11/01,11-15-2026,199.99,149.99,Electronics,TechCorp,High-speed wireless router with dual-band support,NaN,200
1,SKU_FASH_001,Summer Breeze Dress,11/05/2026,2026.11.20,49.99,29.99,Apparel,FashionCo,Summer floral dress in breathable cotton,NaN,220
2,SKU_BEAUTY_001,YouthRevive Night Cream,"Nov 10, 2026",25-Nov-26,89.00,75.00,Beauty,FashionCo,Anti-aging night cream enriched with hyaluroni...,50 ml,220
3,SKU_BEVERAGE_001,MorningKick Cold Brew,11/11/26,2026-11-12,12.00,8.00,Groceries,TechCorp,Organic cold brew coffee ready to drink,16 oz,200
4,SKU_HOME_001,GreenEarth Laundry Detergent,2026-11-20 00:00:00,2026-11-30T23:59:59Z,29.99,19.99,Home,FashionCo,Eco-friendly liquid laundry detergent,2 L,220


### 2. Generate Promotional Copy (Testing One Item)
Let's test the workflow on the very first row.

##### load the style

In [12]:
def load_style_guides(styles_dir):
    styles = {}
    for filename in os.listdir(styles_dir):
        if filename.endswith('_style.md'):
            client = filename.replace('_style.md', '')
            with open(os.path.join(styles_dir, filename), 'r') as f:
                styles[client] = f.read()
    return styles

styles = load_style_guides('styles')

### generate prompt

In [13]:
# Get the first row
row = df.iloc[0]
sku = row['sku']
client = row['client']
style_guide = styles.get(client, "")

print(f"Processing SKU: {sku} for Client: {client}\n")

# Build the Prompt for a Simplified Copy
prompt = f"""
You are an expert copywriter. Please write a SIMPLIFIED promotional copy for the following product.

Product Details:
- SKU: {sku}
- Title: {row['title']}
- Description: {row['description']}
- Original Price: {row['orig_price']}
- Promo Price: {row['promo_price']}
- Max Characters Allowed: {row['max_copy_chars']}

Brand Style Guide:
{style_guide}

Output ONLY the copy, nothing else.
"""


Processing SKU: SKU_TECH_001 for Client: TechCorp



#### generate copy

#### llm.call

In [14]:
# Call the LLM (This will use Mock if Anthropic is not set up)
# generated_copy = llm.call(prompt)
generated_copy = llm.call([{"role": "user", "content": prompt}])
print("=== Generated Copy ===")
print(generated_copy)
print("======================")

=== Generated Copy ===
Save $50 on TechNet Dual-Band Router. High-speed wireless performance at **$149.99**. Was $199.99.


#### llm.structured_call

In [15]:
from schema import PromoItem

In [16]:
structured_output = llm.structured_call(
    messages=[{"role": "user", "content": prompt}],
    output_format=PromoItem  # 直接传 Pydantic 模型
)



In [17]:
print(structured_output)

brand=None product_name='Dual-Band Router' size_value=None size_unit=None pack_count=1 deal_type='sale_price' deal_qty=1 deal_price=149.99 reg_price=199.99 save_amount=50.0


In [18]:
df.iloc[0]

sku                                                    SKU_TECH_001
title                                      TechNet Dual-Band Router
promo_start                                              2026/11/01
promo_end                                                11-15-2026
orig_price                                                   199.99
promo_price                                                  149.99
category                                                Electronics
client                                                     TechCorp
description       High-speed wireless router with dual-band support
size                                                            NaN
max_copy_chars                                                  200
Name: 0, dtype: object

### 3. Verification & Evaluation
Two-layer validation:
1. **Rule-based checks** — deterministic hard constraints (char limit, price, emoji, formatting).
2. **LLM-as-Judge** — subjective quality evaluation via `structured_chat` (tool-use), guaranteed JSON schema.

##### rule checks

In [19]:
import re

def rule_checks(copy_text: str, row: dict, client: str) -> dict:
    """
    第一层验证：规则检查（确定性检查）
    """
    result = {}
    
    # 1. 通用检查
    result['length_ok'] = len(copy_text) <= row.get('max_copy_chars', 150)
    result['char_count'] = len(copy_text)
    result['price_mentioned'] = any(c in copy_text.lower() for c in ['$', 'price', 'save'])
    
    # 2. 检测 emoji（Unicode 范围）
    emoji_pattern = r'[\U0001F300-\U0001FAFF]'
    has_emoji = bool(re.search(emoji_pattern, copy_text))
    
    # 3. 客户端特定规则
    if client == "TechCorp":
        result['price_bolded'] = '**$' in copy_text  # 价格必须加粗
        result['no_emoji'] = not has_emoji  # 禁止 emoji
    
    elif client == "FashionCo":
        result['has_emoji'] = has_emoji  # 必须有 emoji
        result['price_has_exclamation'] = re.search(r'\$[\d.]+!', copy_text) is not None
    
    # 4. 总体判断
    if client == "TechCorp":
        required_checks = ['length_ok', 'price_mentioned', 'price_bolded', 'no_emoji']
    else:  # FashionCo
        required_checks = ['length_ok', 'price_mentioned', 'has_emoji', 'price_has_exclamation']
    
    result['all_rules_pass'] = all(result.get(k, False) for k in required_checks)
    result['failed_rules'] = [k for k in required_checks if not result.get(k, False)]
    
    return result

##### LLM-as-a-judge

In [20]:
from src.llm import llm

def llm_judge(copy_text, row, style_guide):
    prompt = f"""You are a QA reviewer evaluating promotional copy.

Product: {row.get('product_name', '')}
Copy to evaluate: {copy_text}
Max characters: {row.get('max_copy_chars', 'N/A')}

Style guide constraints:
{style_guide}

Evaluate across these dimensions and provide scores (1-5) and reasoning for each."""

    return llm.structured_call(
        messages=[{'role': 'user', 'content': prompt}],
        output_format=JudgeEvaluation,  # Pass the Pydantic class here
        model="claude-opus-5-5",
        max_tokens=1024
    )

#### verify all items

In [21]:
def verify_copy(copy_text: str, row: dict, client_type: str = "TechCorp", lang: str = "en") -> dict:
    """
    完整的两层验证
    """
    
    # Layer 1: 规则检查
    rule_result = rule_checks(copy_text, row, client_type)
    
    # Layer 2: LLM 评估
    llm_result = llm_judge(copy_text, row, client_type)
    
    # 综合判断
    final_pass = rule_result['all_rules_pass'] and llm_result.overall_pass
    
    return {
        'sku': row.get('sku'),
        'copy': copy_text,
        'rule_checks': rule_result,
        'llm_judge': llm_result,
        'final_pass': final_pass,
        'status': '✅ PASS' if final_pass else '❌ FAIL'
    }

In [ ]:
from schema import JudgeEvaluation
# 假设你已经有生成的文案
copies = [
    {
        'sku': 'TECH001',
        'text': 'Save on **$149.99** wireless router - limited time',
        'client': 'TechCorp'
    },
    {
        'sku': 'FASH001', 
        'text': 'Summer floral dress ✨ Now $29.99!',
        'client': 'FashionCo'
    }
]

results = []
for copy in copies:
    verification = verify_copy(
        copy_text=copy['text'],
        row={'sku': copy['sku'], 'max_copy_chars': 150, 'product_name': 'Product'},
        client_type=copy['client']
    )
    results.append(verification)
    print(f"{verification['status']} - {copy['sku']}")
    print(f"  规则检查: {verification['rule_checks']['all_rules_pass']}")
    print(f"  LLM 评估: {verification['llm_judge'].overall_pass}")

❌ FAIL - TECH001
  规则检查: True


TypeError: 'JudgeEvaluation' object is not subscriptable